# Phase 2 — Degraded Planner & Operator Metrics

Phase 1 gave us: assign → simulate → replan → visualize.

Phase 2 answers the harder question:
> **When something fails, how screwed are we?**

New in this notebook:
1. **Three-tier planner** — Full MILP → Degraded MILP → Heuristic fallback
2. **Battery-constrained MILP** — optimizer plans around range limits, not just discovers them
3. **Operator metrics** — time-to-recovery, coverage-at-failure (not optimality gaps)
4. **Monte Carlo worst-case analysis** — distribution of outcomes across 100 random failure scenarios
5. **Planner mode comparison** — does degraded mode cost us anything?

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.field.generator import synthetic_field, generate_strips
from src.optimizer.milp import DroneSpec, assign_strips
from src.optimizer.planner import plan, PlannerMode, PlannerContext, select_mode
from src.simulation.engine import simulate
from src.simulation.metrics import (
    compute_metrics, compare_runs,
    monte_carlo_analysis,
    plot_coverage_over_time, plot_battery_over_time,
    plot_monte_carlo, plot_metrics_bar,
)

%matplotlib inline

NROWS, NCOLS = 8, 8
N_DRONES = 3

field_grid = synthetic_field(nrows=NROWS, ncols=NCOLS, seed=42)
strips     = generate_strips(field_grid, seconds_per_cell=2.0)
drones     = [DroneSpec(id=i, battery=100.0, spray_capacity=100.0) for i in range(N_DRONES)]

print(f'{len(strips)} strips, {N_DRONES} drones')

## 1. Three-Tier Planner

The planner selects its mode from context. It can also be forced.

| Mode | When used | How |
|---|---|---|
| **Full** | Stable state, sufficient time | Full MILP, all strips, all drones |
| **Degraded** | Time pressure, high uncertainty, low battery | Filtered drones, limited horizon, shorter solve budget |
| **Heuristic** | Timeout, emergency, solver failure | Greedy priority-weighted round-robin |

The mode is chosen automatically from a `PlannerContext` — or forced explicitly.

In [ ]:
# Auto mode selection based on context
contexts = {
    'Stable, 30s budget':       PlannerContext(time_budget_seconds=30.0, uncertainty=0.0),
    'Tight, 1.5s budget':       PlannerContext(time_budget_seconds=1.5,  uncertainty=0.0),
    'High uncertainty (0.8)':   PlannerContext(time_budget_seconds=30.0, uncertainty=0.8),
    'Moderate, 7s budget':      PlannerContext(time_budget_seconds=7.0,  uncertainty=0.3),
}

print(f'{"Context":<30}  {"Auto-selected mode"}')
print('-' * 55)
for label, ctx in contexts.items():
    mode = select_mode(len(strips), N_DRONES, ctx)
    print(f'{label:<30}  {mode.value}')

In [ ]:
# Run all three modes and compare results
results = {}
for mode in PlannerMode:
    r = plan(strips, drones, objective_mode='makespan', mode=mode)
    results[mode.value] = r
    print(f'--- {mode.value.upper()} ---')
    print(f'  Status: {r.status}  Makespan: {r.makespan}s  Solve time: {r.solve_time}s')
    for d_id, sids in r.assignment.items():
        workload = sum(s.time for s in strips if s.id in sids)
        print(f'  Drone {d_id}: strips={sids}  workload={workload:.1f}s')
    print()

## 2. Battery-Constrained MILP

Phase 1 discovered battery limits *during simulation*. Now the optimizer *plans around* them.

`battery_capacity_seconds` sets a hard per-drone workload ceiling in the MILP.
If strips can't all be covered within that limit, the solver returns **Infeasible** —
a signal to the operator that more drones or a different field partition is needed.

This is the correct failure mode: explicit planning-time infeasibility beats
silent mid-mission collapse.

In [ ]:
total_time = sum(s.time for s in strips)
per_drone_unconstrained = total_time / N_DRONES
print(f'Total strip time: {total_time:.1f}s')
print(f'Unconstrained per-drone avg: {per_drone_unconstrained:.1f}s')
print()

# Try three battery caps: generous, tight, infeasible
caps = {
    f'Unconstrained':                None,
    f'Generous ({per_drone_unconstrained*1.3:.0f}s cap)':  per_drone_unconstrained * 1.3,
    f'Tight ({per_drone_unconstrained*0.9:.0f}s cap)':     per_drone_unconstrained * 0.9,
    f'Infeasible ({per_drone_unconstrained*0.5:.0f}s cap)': per_drone_unconstrained * 0.5,
}

for label, cap in caps.items():
    r = plan(strips, drones, battery_capacity_seconds=cap, mode=PlannerMode.FULL)
    if r.status in ('Optimal', 'Feasible'):
        workloads = {d_id: sum(s.time for s in strips if s.id in sids)
                     for d_id, sids in r.assignment.items()}
        print(f'{label}: status={r.status}  makespan={r.makespan}s  max_workload={max(workloads.values()):.1f}s')
    else:
        print(f'{label}: status={r.status} (cap too tight — need more drones or re-partition)')

## 3. Operator Metrics: Time-to-Recovery

When a drone fails:
- **Coverage-at-failure**: how far along were we?
- **Time-to-recovery**: how many steps until coverage started growing again?

These are the metrics operators at Zipline or DJI care about —
not optimality gaps.

In [ ]:
baseline_result = plan(strips, drones, mode=PlannerMode.FULL)

# Drone 0 fails early; drone 1 gets a comms dropout
failure_events = [
    {'timestep': 8,  'drone_id': 0, 'type': 'battery'},
    {'timestep': 20, 'drone_id': 1, 'type': 'comms', 'duration': 5},
]

hist_fail = simulate(
    strips=strips, drones=drones, result=baseline_result,
    nrows=NROWS, ncols=NCOLS,
    failure_events=failure_events,
)
m_fail = compute_metrics(hist_fail, strips, NROWS, NCOLS)

print('Failure run metrics:')
for k in ['coverage_pct', 'priority_coverage', 'makespan',
          'replan_count', 'failed_drone_count',
          'coverage_at_failure', 'time_to_recovery']:
    print(f'  {k:<25}: {m_fail[k]}')

print('\nEvent log:')
for s in hist_fail:
    if s['event']:
        print(f'  t={s["timestep"]:3d}: {s["event"]}')

In [ ]:
hist_base = simulate(
    strips=strips, drones=drones, result=baseline_result,
    nrows=NROWS, ncols=NCOLS,
)
m_base = compute_metrics(hist_base, strips, NROWS, NCOLS)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_coverage_over_time(
    {'Baseline': m_base['cells_per_step'], 'With failures': m_fail['cells_per_step']},
    total_cells=NROWS * NCOLS,
    title='Coverage over time',
    ax=axes[0],
)
# Mark failure and recovery
if m_fail['failure_steps']:
    axes[0].axvline(m_fail['failure_steps'][0], color='red', linestyle=':', alpha=0.8, label='First failure')
    axes[0].legend()

plot_battery_over_time(
    battery_series=m_fail['battery_series'],
    replan_steps=m_fail['replan_steps'],
    title='Battery over time (failure run)',
    ax=axes[1],
)

plt.tight_layout()
plt.show()

## 4. Planner Mode Comparison Under Failure

Does using degraded or heuristic replanning cost us coverage?
Run the same failure scenario three times — replanning with different modes.

In [ ]:
mode_histories = {}
mode_metrics   = {}

for replan_mode in ['makespan', 'weighted']:
    for planner_mode in [PlannerMode.FULL, PlannerMode.DEGRADED, PlannerMode.HEURISTIC]:
        label = f'{planner_mode.value}/{replan_mode}'

        # Initial plan always uses full MILP
        initial = plan(strips, drones, objective_mode=replan_mode, mode=PlannerMode.FULL)

        hist = simulate(
            strips=strips, drones=drones, result=initial,
            nrows=NROWS, ncols=NCOLS,
            failure_events=[{'timestep': 8, 'drone_id': 0, 'type': 'battery'}],
            replan_objective=replan_mode,
        )
        m = compute_metrics(hist, strips, NROWS, NCOLS)
        mode_histories[label] = m['cells_per_step']
        mode_metrics[label]   = m

comp = compare_runs(
    mode_metrics,
    keys=['coverage_pct', 'priority_coverage', 'makespan', 'time_to_recovery'],
)
print(f'{"Metric":<25}', '  '.join(f'{k:<22}' for k in mode_metrics))
print('-' * 145)
for metric, vals in comp.items():
    row = '  '.join(f'{v!s:<22}' for v in vals.values())
    print(f'{metric:<25}{row}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_coverage_over_time(
    mode_histories,
    total_cells=NROWS * NCOLS,
    title='Coverage over time — planner mode comparison (drone 0 fails at t=8)',
    ax=ax,
)
plt.tight_layout()
plt.show()

## 5. Monte Carlo Worst-Case Analysis

Run 100 simulations with randomly injected failures and random battery drain.
This answers: **what's the realistic worst case?**

The P5 coverage is what an operator should plan around — not the mean.

In [ ]:
initial_result = plan(strips, drones, mode=PlannerMode.FULL)

mc_full = monte_carlo_analysis(
    strips, drones, initial_result, NROWS, NCOLS,
    n_runs=100,
    failure_prob_per_drone=0.35,
    battery_drain_range=(0.0, 5.0),
    replan_objective='makespan',
    seed=42,
)

mc_weighted = monte_carlo_analysis(
    strips, drones,
    plan(strips, drones, objective_mode='weighted', mode=PlannerMode.FULL),
    NROWS, NCOLS,
    n_runs=100,
    failure_prob_per_drone=0.35,
    battery_drain_range=(0.0, 5.0),
    replan_objective='weighted',
    seed=42,
)

print('Monte Carlo results (100 runs, failure_prob=0.35/drone, drain=0-5%/cell):')
print()
print(f'{"Metric":<30}  {"Makespan plan":>16}  {"Weighted plan":>16}')
print('-' * 68)
for key in ['coverage_pct_mean', 'coverage_pct_std', 'coverage_pct_p5',
            'coverage_pct_p95', 'priority_coverage_mean', 'priority_coverage_p5',
            'time_to_recovery_mean', 'time_to_recovery_p95']:
    v1 = mc_full.get(key)
    v2 = mc_weighted.get(key)
    print(f'{key:<30}  {str(v1):>16}  {str(v2):>16}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_monte_carlo(mc_full,     title='Coverage distribution — Makespan plan',  ax=axes[0])
plot_monte_carlo(mc_weighted, title='Coverage distribution — Weighted plan',   ax=axes[1])

plt.tight_layout()
plt.show()

print(f'Makespan plan  P5={mc_full["coverage_pct_p5"]}%  (worst-case planning target)')
print(f'Weighted plan  P5={mc_weighted["coverage_pct_p5"]}%')
print()
print('Priority coverage P5 (what matters when batteries run out):')
print(f'  Makespan : {mc_full["priority_coverage_p5"]:.3f}')
print(f'  Weighted : {mc_weighted["priority_coverage_p5"]:.3f}')

## 6. Sensitivity: Failure Rate vs. Coverage

Sweep failure probability from 0 to 0.8 and plot how P5 coverage degrades.
This is the kind of analysis an operator would use to decide how many backup drones to deploy.

In [ ]:
initial_result = plan(strips, drones, mode=PlannerMode.FULL)
probs = np.linspace(0.0, 0.8, 9)

mean_covs, p5_covs = [], []

for prob in probs:
    mc = monte_carlo_analysis(
        strips, drones, initial_result, NROWS, NCOLS,
        n_runs=60, failure_prob_per_drone=float(prob),
        battery_drain_range=(0.0, 3.0), seed=99,
    )
    mean_covs.append(mc['coverage_pct_mean'])
    p5_covs.append(mc['coverage_pct_p5'])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(probs, mean_covs, label='Mean coverage', marker='o')
ax.plot(probs, p5_covs,  label='P5 coverage (worst-case)', marker='s', linestyle='--')
ax.fill_between(probs, p5_covs, mean_covs, alpha=0.15)
ax.set_xlabel('Failure probability per drone')
ax.set_ylabel('Coverage (%)')
ax.set_title('Coverage vs. failure rate (60 MC runs each)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| What we built | Why it matters |
|---|---|
| Three-tier planner | System stays functional when MILP is too slow |
| Battery-constrained MILP | Infeasibility at planning time > silent mid-mission collapse |
| Time-to-recovery metric | Quantifies the cost of a failure event |
| Monte Carlo worst-case | P5 coverage is what operators should plan around, not the mean |
| Sensitivity sweep | Tells operators how many backup drones they need |

**Next: Phase 3** — real field data from the cropland dataset, battery-aware routing heuristic, Streamlit dashboard.